# Plot the Results of Model Evaluations

In [ ]:
from pathlib import Path

from adjustText import adjust_text
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from ultrabench.models.echocare.echocare_backbone import EchoCareBackbone
from ultrabench.models.samus.samus_backbone import SAMUSBackbone
from ultrabench.models.ultrasam.ultrasam_backbone import UltraSAMBackbone
from ultrabench.models.urfm.urfm_backbone import URFMBackbone
from ultrabench.models.usfm.usfm_backbone import USFMBackbone
from ultrabench.models.usfmae.usfmae_backbone import USFMAEBackbone
from ultrabench.models.deblurmim.deblurmim_backbone import DeblurMIMBackbone
from ultrabench.models.dinov3.dinov3_backbone import (
    DINOv3ViTBackbone,
    DINOv3ConvNeXtBackbone,
)
from ultrabench.models.openus.openus_backbone import OpenUSBackbone
from ultrabench.models.ultrafedfm.ultrafedfm_backbone import UltraFedFMBackbone

SWEEPS_DIR = Path("/project/ultrabench2/runs/")
MODEL_DISPLAY_NAMES = {
    "deblurmim_convmae": "Deblurring ConvMAE",
    "deblurmim_mae": "Deblurring MAE",
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "dinov3_vitb16": "DINOv3 ViT-B",
    "dinov3_vitl16": "DINOv3 ViT-B",
    "echocare": "EchoCare",
    "openus": "OpenUS",
    "samus": "SAMUS",
    "ultrafedfm": "UltraFedFM",
    "ultrasam": "UltraSAM",
    "urfm_base": "URFM-B",
    "urfm_large": "URFM-L",
    "usfm": "USFM",
    "usfmae": "USF-MAE",
}

# Configure plotting libraries
plt.rcParams["figure.figsize"] = (6.5, 2.48)
sns.set_theme(style="whitegrid", context="paper", palette="colorblind", font_scale=0.8)

## Load Aggregated Test Results

Load the `test_results.csv` file generated by `aggregate_results.ipynb`.

In [ ]:
test_results = pd.read_csv("test_results.csv")
test_results.head()

## Performance vs. Release Date

In [ ]:
# The date of the publication of the FIRST version of the paper describing the
# model on arXiv.
release_dates = {
    "deblurmim_convmae": "2023-06-14",
    "deblurmim_mae": "2023-06-14",
    "dinov3_convnext_base": "2025-08-13",
    "dinov3_convnext_large": "2025-08-13",
    "dinov3_vitb16": "2025-08-13",
    "dinov3_vitl16": "2025-08-13",
    "echocare": "2025-09-15",
    "openus": "2025-11-14",
    "samus": "2023-09-13",
    "ultrafedfm": "2024-11-25",
    "ultrasam": "2024-11-25",
    "urfm_base": "2025-08-15",
    "urfm_large": "2025-08-15",
    "usfm": "2023-12-30",
    "usfmae": "2025-10-27",
}

In [ ]:
# Group the classification results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_acc`
# for each group
classification_test_results = test_results[
    test_results["task"].str.contains("classification")
]

classification_test_summary = (
    classification_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "f1_macro": ["mean"],
        }
    )
    .reset_index()
)

# Flatten the multi-index columns
classification_test_summary.columns = [
    "task",
    "model",
    "shot_pct",
    "mode",
    "macro_f1",
]

classification_test_summary["release_date"] = classification_test_summary["model"].map(
    release_dates
)
classification_test_summary["release_date"] = pd.to_datetime(
    classification_test_summary["release_date"]
)

classification_test_summary.head()

In [ ]:
# Plot overall classification performance by release date
plot_df = (
    classification_test_summary[
        (classification_test_summary["shot_pct"] == 1.0)
        & (classification_test_summary["mode"] == "linear_classifier_full_finetuning")
    ]
    .groupby(["model", "release_date"])
    .agg({"macro_f1": "mean"})
    .reset_index()
)

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="release_date",
    y="macro_f1",
    hue="model",
    ax=ax,
    legend=False,
)

# Add labels for each point
texts = []
for _, row in plot_df.iterrows():
    texts.append(
        ax.text(
            row["release_date"],
            row["macro_f1"],
            MODEL_DISPLAY_NAMES[row["model"]],
            ha="center",
            va="center",
        )
    )
adjust_text(texts)

# Draw a line connecting the current best model to the next best model
sorted_df = plot_df.sort_values(
    by=["release_date", "macro_f1"], ascending=[True, False]
)
current_best_model = sorted_df.iloc[0]
for i in range(1, len(sorted_df)):
    next_model = sorted_df.iloc[i]
    if next_model["macro_f1"] > current_best_model["macro_f1"]:
        ax.plot(
            [current_best_model["release_date"], next_model["release_date"]],
            [current_best_model["macro_f1"], next_model["macro_f1"]],
            color="gray",
            linestyle="--",
            linewidth=0.5,
        )
        current_best_model = next_model

ax.set_ylabel("Mean F1 Score")
ax.set_ylim(0.5, 0.7)

x_labels = ax.get_xticklabels()
x_ticks = ax.get_xticks()
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels)
ax.set_xlabel("Release Date")

plt.tight_layout()
plt.savefig("classification_performance_by_release_date.pdf")
plt.show()

In [ ]:
# Group the segmentation results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_dice`
# for each group
segmentation_test_results = test_results[
    test_results["task"].str.contains("segmentation")
]

segmentation_test_summary = (
    segmentation_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "positive_dice": ["mean"],
        }
    )
    .reset_index()
)

# Flatten the multi-index columns
segmentation_test_summary.columns = [
    "task",
    "model",
    "shot_pct",
    "mode",
    "positive_dice",
]

segmentation_test_summary["release_date"] = segmentation_test_summary["model"].map(
    release_dates
)
segmentation_test_summary["release_date"] = pd.to_datetime(
    segmentation_test_summary["release_date"]
)

segmentation_test_summary.head()

In [ ]:
# Plot overall segmentation performance by release date
plot_df = (
    segmentation_test_summary[
        (segmentation_test_summary["shot_pct"] == 1.0)
        & (segmentation_test_summary["mode"].str.contains("full_finetuning"))
    ]
    .groupby(["model", "release_date"])
    .agg({"positive_dice": "mean"})
    .reset_index()
)

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="release_date",
    y="positive_dice",
    hue="model",
    ax=ax,
    legend=False,
)

# Add labels for each point
texts = []
for _, row in plot_df.iterrows():
    texts.append(
        ax.text(
            row["release_date"],
            row["positive_dice"],
            MODEL_DISPLAY_NAMES[row["model"]],
            ha="center",
            va="center",
        )
    )
adjust_text(texts)

# Draw a line connecting the current best model to the next best model
sorted_df = plot_df.sort_values(
    by=["release_date", "positive_dice"], ascending=[True, False]
)
current_best_model = sorted_df.iloc[0]
for i in range(1, len(sorted_df)):
    next_model = sorted_df.iloc[i]
    if next_model["positive_dice"] > current_best_model["positive_dice"]:
        ax.plot(
            [current_best_model["release_date"], next_model["release_date"]],
            [current_best_model["positive_dice"], next_model["positive_dice"]],
            color="gray",
            linestyle="--",
            linewidth=0.5,
        )
        current_best_model = next_model

ax.set_ylim(0.77, 0.84)
ax.set_ylabel("Mean Positive Dice Score")

x_labels = ax.get_xticklabels()
x_ticks = ax.get_xticks()
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels)
ax.set_xlabel("Release Date")

plt.tight_layout()
plt.savefig("segmentation_performance_by_release_date.pdf")
plt.show()

## Performance vs. Model Size

In [ ]:
from pyexpat import model
from unittest.mock import patch

_BACKBONE_SPECS: list[tuple[str, type, dict]] = [
    (
        "deblurmim_convmae",
        DeblurMIMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "deblurmim_convmae_base",
        },
    ),
    (
        "deblurmim_mae",
        DeblurMIMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "deblurmim_mae_base_patch16",
        },
    ),
    (
        "dinov3_convnext_base",
        DINOv3ConvNeXtBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_convnext_base",
        },
    ),
    (
        "dinov3_convnext_large",
        DINOv3ConvNeXtBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_convnext_large",
        },
    ),
    (
        "dinov3_vitb16",
        DINOv3ViTBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_vitb16",
        },
    ),
    (
        "dinov3_vitl16",
        DINOv3ViTBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_vitl16",
        },
    ),
    (
        "echocare",
        EchoCareBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "openus",
        OpenUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "samus",
        SAMUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": False},
    ),
    (
        "ultrafedfm",
        UltraFedFMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "ultrafedfm_vit_base_patch16",
        },
    ),
    (
        "ultrasam",
        UltraSAMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "urfm",
        URFMBackbone,
        {
            "checkpoint_path": "dummy",
            "model_variant": "urfm_mae_vit_base_patch16",
            "output_intermediate_features": True,
        },
    ),
    (
        "usfm",
        USFMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "usfmae",
        USFMAEBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
]


model_parameter_counts = {}
for model_key, cls, kwargs in _BACKBONE_SPECS:
    if model_key == "dinov3_convnext_base":
        trainable_parameters = 89_000_000
    elif model_key == "dinov3_convnext_large":
        trainable_parameters = 198_000_000
    elif model_key == "dinov3_vitb16":
        trainable_parameters = 86_000_000
    elif model_key == "dinov3_vitl16":
        trainable_parameters = 300_000_000
    else:
        with patch.object(cls, "_load_checkpoint", lambda self: None):
            backbone = cls(**kwargs)

        # Count the number of trainable parameters in the model (excluding
        # "decoder" parameters)
        trainable_parameters = sum(
            p.numel()
            for name, p in backbone.named_parameters()
            if p.requires_grad and "decoder" not in name
        )

        del backbone

    model_parameter_counts[model_key] = trainable_parameters

In [ ]:
# Plot the overall classification performance on the X-axis, the overall
# segmentation performance on the Y-axis, and the number of trainable
# parameters (excluding decoder parameters) as the size of the points.
plot_df = (
    test_results[
        (test_results["shot_pct"] == 1.0)
        & (test_results["mode"].str.contains("full_finetuning"))
    ]
    .groupby(["model"])
    .agg(
        {
            "f1_macro": "mean",
            "positive_dice": "mean",
        }
    )
    .reset_index()
)
plot_df["params"] = plot_df["model"].map(model_parameter_counts)
plot_df["params"] = round(plot_df["params"] / 1e6, 1)  # Convert to millions

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="f1_macro",
    y="positive_dice",
    size="params",
    hue="params",
    ax=ax,
    legend=True,
)

# Add labels for each point
texts = []
for _, row in plot_df.iterrows():
    texts.append(
        ax.text(
            row["f1_macro"],
            row["positive_dice"],
            MODEL_DISPLAY_NAMES[row["model"]],
            ha="center",
            va="center",
        )
    )
adjust_text(texts)

handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{float(label):.1f}M" for label in labels]
plt.legend(handles=handles, labels=new_labels, ncol=2)

ax.set_xlabel("Mean F1 Score")
ax.set_ylabel("Mean Positive Dice Score")

plt.tight_layout()
plt.savefig("performance_vs_model_size.pdf")
plt.show()